# 03b - Week 4 Linear Regression One-Hot Test

Diagnostic rerun of the Week 4 Linear Regression baseline. The purpose is to isolate whether replacing frequency-encoded location variables with one-hot encoding produces validation performance comparable to the other teams.

Mandatory split:
- **Train:** February 2025 through January 2026
- **Validation:** February 2026 through April 2026
- **Test:** May 2026

`ListPrice`, `OriginalListPrice`, and `DaysOnMarket` are excluded. This prevents the encoding test from receiving an artificial advantage from price-related or potentially post-listing information.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
DATA_FILE = Path("data/week5_unstandardized_model_data.csv")
OLD_DATA_FILE = Path("data/week4_cleaned_model_data.csv")
RESULTS_FILE = Path("data/week4_onehot_linear_regression_results.csv")

TRAIN_MONTHS = pd.period_range("2025-02", "2026-01", freq="M").astype(str).tolist()
VALIDATION_MONTHS = pd.period_range("2026-02", "2026-04", freq="M").astype(str).tolist()
TEST_MONTHS = ["2026-05"]

df = pd.read_csv(DATA_FILE, low_memory=False)
old_df = pd.read_csv(OLD_DATA_FILE, low_memory=False)

expected_split_months = {
    "train": TRAIN_MONTHS,
    "validation": VALIDATION_MONTHS,
    "test": TEST_MONTHS,
}
for split_name, expected_months in expected_split_months.items():
    actual = sorted(df.loc[df["split"] == split_name, "close_month"].unique())
    assert actual == expected_months, (split_name, actual, expected_months)

df.groupby("split", sort=False)["close_month"].agg(["min", "max", "nunique", "count"])

,min,max,nunique,count
split,,,,
train,2025-02,2026-01,12,129443
validation,2026-02,2026-04,3,31722
test,2026-05,2026-05,1,12013


## Reproduce the frequency-encoded baseline on the new split

The saved Week 4 `split` column is ignored because it contains the previous date ranges. Rows are selected directly from `close_month`.

In [3]:
old_feature_cols = [
    col for col in old_df.columns
    if col.endswith("_scaled")
    or col.endswith("_freq")
    or col in ["ViewYN", "PoolPrivateYN", "AttachedGarageYN"]
]

old_train = old_df[old_df["close_month"].isin(TRAIN_MONTHS)].copy()
old_validation = old_df[old_df["close_month"].isin(VALIDATION_MONTHS)].copy()
old_test = old_df[old_df["close_month"].isin(TEST_MONTHS)].copy()

frequency_model = LinearRegression()
frequency_model.fit(old_train[old_feature_cols], old_train["ClosePrice"] )

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


## One-hot Linear Regression pipeline

Numeric missing values are imputed using training medians. Numeric scaling is also fitted using training rows only. Scaling does not add predictive information to ordinary least squares, but it is necessary here for stable convergence of the sparse least-squares solver after one-hot encoding.

In [4]:
numeric_features = [
    "Latitude", "Longitude", "LivingArea", "BuildingAreaTotal",
    "BedroomsTotal", "BathroomsTotalInteger", "YearBuilt", "Stories",
    "GarageSpaces", "ParkingTotal", "AssociationFee",
    "LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea",
    "MainLevelBedrooms",
]
boolean_features = [
    "ViewYN", "WaterfrontYN", "PoolPrivateYN", "AttachedGarageYN",
    "FireplaceYN", "NewConstructionYN",
]
categorical_features = [
    "City", "PostalCode", "CountyOrParish", "MLSAreaMajor",
    "HighSchoolDistrict", "Levels",
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("boolean", "passthrough", boolean_features),
    (
        "categorical",
        OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=10,
        ),
        categorical_features,
    ),
])

onehot_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression(n_jobs=-1)),
])

train = df[df["split"] == "train"].copy()
validation = df[df["split"] == "validation"].copy()
test = df[df["split"] == "test"].copy()

onehot_model.fit(train, train["ClosePrice"])
encoded_feature_count = len(onehot_model.named_steps["preprocessor"].get_feature_names_out())
print(f"Encoded feature count: {encoded_feature_count:,}")

Encoded feature count: 2,744


## Compare results

In [5]:
def metric_row(model_name, encoding, split_name, split_df, predictions):
    actual = split_df["ClosePrice"]
    return {
        "model": model_name,
        "encoding": encoding,
        "split": split_name,
        "rows": len(split_df),
        "r2": r2_score(actual, predictions),
        "mae": mean_absolute_error(actual, predictions),
        "rmse": float(np.sqrt(mean_squared_error(actual, predictions))),
    }

rows = []
for split_name, split_df in [
    ("train", old_train),
    ("validation", old_validation),
    ("test", old_test),
]:
    rows.append(metric_row(
        "LinearRegression", "frequency", split_name, split_df,
        frequency_model.predict(split_df[old_feature_cols]),
    ))

for split_name, split_df in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    rows.append(metric_row(
        "LinearRegression", "one_hot", split_name, split_df,
        onehot_model.predict(split_df),
    ))

results = pd.DataFrame(rows)
results.to_csv(RESULTS_FILE, index=False)
results.round({"r2": 4, "mae": 0, "rmse": 0})

,model,encoding,split,rows,r2,mae,rmse
0,LinearRegression,frequency,train,129449,0.5064,480211.0,852237.0
1,LinearRegression,frequency,validation,31724,0.5045,491738.0,885514.0
2,LinearRegression,frequency,test,12013,0.5232,491580.0,839207.0
3,LinearRegression,one_hot,train,129443,0.7966,261880.0,547131.0
4,LinearRegression,one_hot,validation,31722,0.7877,275625.0,579722.0
5,LinearRegression,one_hot,test,12013,0.8097,270148.0,530145.0


In [6]:
validation_comparison = (
    results[results["split"] == "validation"]
    .set_index("encoding")[["r2", "mae", "rmse"]]
)
r2_gain = validation_comparison.loc["one_hot", "r2"] - validation_comparison.loc["frequency", "r2"]

print(validation_comparison.round(4))
print(f"\nValidation R² improvement from one-hot encoding: {r2_gain:.4f}")
print(f"Wrote {RESULTS_FILE}")

               r2          mae         rmse
encoding                                   
frequency  0.5045  491737.9007  885513.5950
one_hot    0.7877  275624.8709  579721.5592

Validation R² improvement from one-hot encoding: 0.2831
Wrote data/week4_onehot_linear_regression_results.csv


## Conclusion

With the mandatory split, one-hot encoding raises Linear Regression validation R² from approximately **0.505** to **0.788**. The May 2026 test R² is approximately **0.810**. This reaches the reported 0.7-0.8 range without using listing-price features.

The evidence indicates that the earlier limitation was primarily the frequency representation of location—not numeric standardization. Week 5 can still use a different representation for tree-based models; this notebook only establishes the corrected Week 4 linear baseline.